### Notebook to prepare the CATNAT dataset

In [2]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [3]:
df_catnat = pd.read_csv('../data/gaspar/catnat_gaspar.csv', sep=';')

df_catnat['dat_deb'] = pd.to_datetime(df_catnat['dat_deb'], format='%Y-%m-%d')
df_catnat['dat_pub_jo'] = pd.to_datetime(df_catnat['dat_pub_jo'], format='%Y-%m-%d')
df_catnat['duree'] = (df_catnat['dat_pub_jo'] - df_catnat['dat_deb']).dt.days

In [4]:
df_catnat[['duree', 'lib_risque_jo']].groupby('lib_risque_jo').agg(['count', 'mean', 'std']).sort_values(('duree', 'count'), ascending=False).head(10)

duree              \
                                                     count        mean   
lib_risque_jo                                                            
Inondations et/ou Coulées de Boue                   145433   70.705369   
Sécheresse                                           46365  826.412984   
Mouvement de Terrain                                 32298   37.989287   
Tempête                                              16187   20.690678   
Chocs Mécaniques liés à l'action des Vagues           6773   25.462720   
Glissement de Terrain                                 3903  107.143992   
Poids de la Neige                                     2758  113.748731   
Mouvements de terrain différentiels consécutifs...    1543  588.787427   
Grêle                                                 1491   56.592220   
Inondations Remontée Nappe                            1430  319.219580   

                                                                
                                                           std  
lib_risque_jo                                                   
Inondations et/ou Coulées de Boue                    87.550285  
Sécheresse                                          715.050420  
Mouvement de Terrain                                130.691360  
Tempête                                              18.063089  
Chocs Mécaniques liés à l'action des Vagues          59.067270  
Glissement de Terrain                               138.837226  
Poids de la Neige                                    95.692776  
Mouvements de terrain différentiels consécutifs...  114.039780  
Grêle                                                20.345375  
Inondations Remontée Nappe                          248.694384

This motivates the need to work in a first time only with floodings.

In [5]:
list_date_election = ['12/06/2022', '11/06/2017','10/06/2012', '10/06/2007', '09/06/2002', '25/05/1997', '21/03/1993']
list_date_election = [pd.to_datetime(date, format='%d/%m/%Y') for date in list_date_election]

#show the number of days between the date of the election and the next election
for i in range(len(list_date_election)-1):
    print((list_date_election[i+1] - list_date_election[i]).days)


-1827
-1827
-1827
-1827
-1841
-1526


In [10]:
df_catnat = df_catnat[df_catnat['lib_risque_jo'] == 'Inondations et/ou Coulées de Boue']
df_catnat = df_catnat[df_catnat['duree'] < 600]
df_final = pd.DataFrame()

for i in range(len(list_date_election) - 1):
    df_temp = df_catnat[(df_catnat['dat_pub_jo'] < list_date_election[i]) & (df_catnat['dat_pub_jo'] > list_date_election[i + 1])].copy()
    one_year = pd.Timedelta(days=365)
    df_temp.loc[:, 'mandat'] = 1
    df_temp.loc[:, 'mandat_1'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - one_year))
    df_temp.loc[:, 'mandat_2'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - 2 * one_year))
    df_temp.loc[:, 'mandat_3'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - 3 * one_year))
    df_temp.loc[:, 'mandat_4'] = ((list_date_election[i] - df_temp['dat_pub_jo']) < (list_date_election[i] - list_date_election[i + 1] - 4 * one_year))
    df_temp = df_temp[['cod_commune', 'lib_commune', 'mandat', 'mandat_1', 'mandat_2', 'mandat_3', 'mandat_4']].groupby(['cod_commune', 'lib_commune']).sum().reset_index()
    df_temp.loc[:, 'election'] = list_date_election[i]
    df_final = pd.concat([df_final, df_temp], axis=0)

In [11]:
df_final.to_csv('../data/catnat_gaspar_mandat.csv', sep=';', index=False)

In [12]:
df_final[df_final.lib_commune == 'Montpellier']

,cod_commune,lib_commune,mandat,mandat_1,mandat_2,mandat_3,mandat_4,election
2730,34172,Montpellier,2,2,2,1,1,2022-06-12
2336,34172,Montpellier,7,7,7,2,1,2017-06-11
4313,34172,Montpellier,1,1,1,0,0,2012-06-10
1893,34172,Montpellier,4,3,1,1,0,2007-06-10
9948,34172,Montpellier,2,1,1,1,1,2002-06-09
4615,34172,Montpellier,2,1,0,0,0,1997-05-25
